# Colab Runner for Intersection Traffic Analytics

This notebook mounts Google Drive, clones the GitHub repo, links large camera / LiDAR data into the repo layout, and runs the existing camera-only and fusion workflows without putting the raw files into GitHub.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/YOUR_USERNAME/intersection-traffic-analytics.git'
WORKDIR = Path('/content/intersection-traffic-analytics')

if WORKDIR.exists():
    %cd /content/intersection-traffic-analytics
    subprocess.run(['git', 'pull'], check=True)
else:
    %cd /content
    subprocess.run(['git', 'clone', REPO_URL], check=True)
    %cd /content/intersection-traffic-analytics

In [ ]:
%cd /content/intersection-traffic-analytics
!pip install -r requirements.txt

## Set Your Data Paths

- `SCENE_NAME` should match the scene geometry you want to use, for example `intersection_demo`.
- `CAMERA_SOURCE` should point to the large video file in Drive.
- `LIDAR_SOURCE` can point either to a precomputed `evidence.csv` file or to a raw LiDAR directory.
- `OUTPUT_ROOT` is where you want generated results saved.

In [ ]:
SCENE_NAME = 'intersection_demo'
CAMERA_SOURCE = '/content/drive/MyDrive/traffic_data/intersection_demo.mp4'
LIDAR_SOURCE = '/content/drive/MyDrive/traffic_data/intersection_demo_evidence.csv'  # optional
OUTPUT_ROOT = '/content/drive/MyDrive/traffic_analytics_outputs'
USE_COPY = False  # keep False in Colab unless you really want to duplicate files

In [ ]:
%cd /content/intersection-traffic-analytics

import subprocess

cmd = [
    'python',
    'scripts/link_external_data.py',
    '--scene', SCENE_NAME,
    '--camera-source', CAMERA_SOURCE,
    '--force',
]
if LIDAR_SOURCE:
    cmd.extend(['--lidar-source', LIDAR_SOURCE])
if USE_COPY:
    cmd.append('--copy')
print(' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
%cd /content/intersection-traffic-analytics
!python scripts/export_scene_preview.py --scene configs/scenes/{SCENE_NAME}.yaml --frame-index 960

In [ ]:
%cd /content/intersection-traffic-analytics
!python scripts/compare_trackers.py --scene configs/scenes/{SCENE_NAME}.yaml --output-root "{OUTPUT_ROOT}"

In [ ]:
%cd /content/intersection-traffic-analytics
!python scripts/run_fusion_experiments.py --scenes {SCENE_NAME} --output-root "{OUTPUT_ROOT}" --make-plots

## Notes

- If `LIDAR_SOURCE` is a real `evidence.csv`, the fusion run will use it.
- If no LiDAR evidence file is linked, `run_fusion_experiments.py` falls back to mock motion-based evidence unless you add `--no-mock-lidar`.
- The current fusion MVP expects precomputed LiDAR evidence rather than raw point clouds directly.